# Phase D Full-Run Gate (Kaggle H100)

- Change class: `capability_expansion`
- Phase target: `D`
- Run mode: `full-run only` (no sampling)
- References: `notebook/h100_validation_kaggle.ipynb`, `notebook/phase_c_h100_pinned_manifest.ipynb`

This notebook bootstraps a Kaggle H100 runtime, rebuilds pinned-manifest S8 paths, and runs `scripts/phase_d_gate.py` with full ConceptARC + re_arc diagnostics.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_INPUT_DIR = Path('/kaggle/input/datasets/danielwuu/iris-main/packages')
WHEEL_DIR = Path('/kaggle/input/datasets/danielwuu/wheels/wheels/linux_cp312')
WORK_REPO = Path('/kaggle/working/IRIS')

ARTIFACT_ROOT = WORK_REPO / 'artifacts' / 'phase_d_h100_fullrun'
PHASE_ROOT = ARTIFACT_ROOT / 'phase_root'
BASELINE_DIR = ARTIFACT_ROOT / 'baseline_phase_d_v1'
REPORT_FREEZE_DIR = ARTIFACT_ROOT / 'report_freeze'
REPORT_DIFF_DIR = ARTIFACT_ROOT / 'report_diff'
PINNED_MANIFEST = ARTIFACT_ROOT / 'runtime_lock_manifest_pinned.json'

H100_UNINTERRUPTED = ARTIFACT_ROOT / 'toy_train_gpu_h100'
H100_EXECUTE = ARTIFACT_ROOT / 'toy_resume_h100'
H100_PRE_COMMIT = ARTIFACT_ROOT / 'toy_resume_h100_pre_commit'
H100_POST_COMMIT = ARTIFACT_ROOT / 'toy_resume_h100_post_commit'

DEVICE = 'gpu'
STRICT_JAX = True
MAX_REASONING_CYCLES = 1
TERMINATION_THRESHOLD = 0.5
SEED = 17


def run_shell(cmd: str, cwd: Path | None = None, ok: tuple[int, ...] = (0,), env: dict | None = None):
    print(f'$ {cmd}')
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(proc.stdout)
    if proc.returncode not in ok:
        raise RuntimeError(f'command failed ({proc.returncode}): {cmd}')
    return proc

print('python:', sys.version)
print('repo input exists:', REPO_INPUT_DIR.exists(), REPO_INPUT_DIR)
print('wheel dir exists:', WHEEL_DIR.exists(), WHEEL_DIR)


In [ ]:
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('This notebook expects Python 3.12 because wheels are cp312.')
if not REPO_INPUT_DIR.exists():
    raise FileNotFoundError(f'Missing repo input dir: {REPO_INPUT_DIR}')
if not WHEEL_DIR.exists():
    raise FileNotFoundError(f'Missing wheel dir: {WHEEL_DIR}')

if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)
shutil.copytree(REPO_INPUT_DIR, WORK_REPO)

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('copied repo to:', WORK_REPO)
print('artifact root:', ARTIFACT_ROOT)


In [ ]:
run_shell(f"{sys.executable} -m pip install --no-index --find-links {WHEEL_DIR} --upgrade pip")


def wheel_version(path: Path) -> str:
    return path.name.split('-')[1]


jax_whls = sorted(WHEEL_DIR.glob('jax-*.whl'))
jaxlib_whls = sorted(WHEEL_DIR.glob('jaxlib-*.whl'))
if not jax_whls or not jaxlib_whls:
    raise RuntimeError('Missing jax/jaxlib wheels in wheel directory.')

jax_by_version = {wheel_version(p): p for p in jax_whls}
jaxlib_by_version = {wheel_version(p): p for p in jaxlib_whls}
common_versions = sorted(set(jax_by_version) & set(jaxlib_by_version))
if not common_versions:
    raise RuntimeError('No common jax/jaxlib version found in wheels.')

v = common_versions[-1]
plugin_by_version = {wheel_version(p): p for p in sorted(WHEEL_DIR.glob('jax_cuda12_plugin-*.whl'))}
pjrt_by_version = {wheel_version(p): p for p in sorted(WHEEL_DIR.glob('jax_cuda12_pjrt-*.whl'))}

wheels = [jax_by_version[v], jaxlib_by_version[v]]
if v in plugin_by_version and v in pjrt_by_version:
    wheels.extend([plugin_by_version[v], pjrt_by_version[v]])

wheel_args = ' '.join(str(w) for w in wheels)
run_shell(f"{sys.executable} -m pip install --no-index --find-links {WHEEL_DIR} {wheel_args}")


In [ ]:
run_shell('nvidia-smi -L')
run_shell('nvidia-smi')

import jax
print('jax version:', jax.__version__)
print('jax devices:', jax.devices())
if not any(dev.platform == 'gpu' for dev in jax.devices()):
    raise RuntimeError('No JAX GPU device detected on this Kaggle runtime.')


In [ ]:
RUN_ENV = os.environ.copy()
RUN_ENV.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
RUN_ENV.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.85')

PYTHON = Path(sys.executable)
ROOT = WORK_REPO


def run(args, check=True):
    cmd = [str(PYTHON)] + args
    print('$ ' + ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, env=RUN_ENV)
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed rc={proc.returncode}')
    return proc


def strict_jax_flags():
    return ['--device', DEVICE] + (['--strict-jax'] if STRICT_JAX else ['--no-strict-jax'])


def run_crash_resume(output_dir: Path, crash_point: str, resume_path_id: str):
    shutil.rmtree(output_dir, ignore_errors=True)
    run([
        'scripts/train_toy.py',
        '--output-dir', str(output_dir),
        '--segments', '1',
        '--crash-point', crash_point,
        '--crash-segment', '0',
        '--runtime-lock-manifest', str(PINNED_MANIFEST),
    ] + strict_jax_flags(), check=False)
    run([
        'scripts/train_toy.py',
        '--output-dir', str(output_dir),
        '--segments', '1',
        '--resume-path-id', resume_path_id,
        '--runtime-lock-manifest', str(PINNED_MANIFEST),
    ] + strict_jax_flags())


In [ ]:
# 1) Bootstrap one pinned runtime lock manifest.
seed_dir = ARTIFACT_ROOT / 'runtime_lock_seed'
shutil.rmtree(seed_dir, ignore_errors=True)
run([
    'scripts/train_toy.py',
    '--output-dir', str(seed_dir),
    '--segments', '0',
] + strict_jax_flags())

PINNED_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(seed_dir / 'runtime_lock_manifest.json', PINNED_MANIFEST)
print('Pinned manifest:', PINNED_MANIFEST)


In [ ]:
# 2) Preflight suites.
run(['scripts/s1_smoke.py', '--device', DEVICE])
run(['scripts/s2_structural.py'])
run(['scripts/s2_mounted.py', '--device', DEVICE])


In [ ]:
# 3) Rebuild local S8 packet under PHASE_ROOT.
for name in ['s8_uninterrupted', 's8_execute', 's8_pre_commit', 's8_post_commit']:
    shutil.rmtree(PHASE_ROOT / name, ignore_errors=True)

run([
    'scripts/train_toy.py',
    '--output-dir', str(PHASE_ROOT / 's8_uninterrupted'),
    '--segments', '1',
    '--resume-path-id', 'uninterrupted',
    '--runtime-lock-manifest', str(PINNED_MANIFEST),
] + strict_jax_flags())
run_crash_resume(PHASE_ROOT / 's8_execute', 'execute', 'execute_crash')
run_crash_resume(PHASE_ROOT / 's8_pre_commit', 'pre_commit', 'pre_commit_crash')
run_crash_resume(PHASE_ROOT / 's8_post_commit', 'post_commit', 'post_commit_crash')


In [ ]:
# 4) Rebuild H100 packet paths + model run directory (single source for phase_d_gate).
for path in [H100_UNINTERRUPTED, H100_EXECUTE, H100_PRE_COMMIT, H100_POST_COMMIT]:
    shutil.rmtree(path, ignore_errors=True)

run([
    'scripts/train_toy.py',
    '--output-dir', str(H100_UNINTERRUPTED),
    '--segments', '1',
    '--resume-path-id', 'uninterrupted',
    '--runtime-lock-manifest', str(PINNED_MANIFEST),
] + strict_jax_flags())
run_crash_resume(H100_EXECUTE, 'execute', 'execute_crash')
run_crash_resume(H100_PRE_COMMIT, 'pre_commit', 'pre_commit_crash')
run_crash_resume(H100_POST_COMMIT, 'post_commit', 'post_commit_crash')

print('model_run_dir =', H100_UNINTERRUPTED)


In [ ]:
# 5) Full-run baseline freeze (phase-d-v1).
shutil.rmtree(BASELINE_DIR, ignore_errors=True)
shutil.rmtree(REPORT_FREEZE_DIR, ignore_errors=True)

freeze_proc = run([
    'scripts/phase_d_gate.py',
    '--phase', 'D',
    '--device', DEVICE,
    '--model-run-dir', str(H100_UNINTERRUPTED),
    '--phase-root', str(PHASE_ROOT),
    '--baseline-report-dir', str(BASELINE_DIR),
    '--output-dir', str(REPORT_FREEZE_DIR),
    '--freeze-baseline',
    '--conceptarc-corpus', 'tools/ConceptARC/corpus',
    '--rearc-tasks', 'data/arc/re_arc/tasks',
    '--h100-uninterrupted', str(H100_UNINTERRUPTED),
    '--h100-execute', str(H100_EXECUTE),
    '--h100-pre-commit', str(H100_PRE_COMMIT),
    '--h100-post-commit', str(H100_POST_COMMIT),
    '--max-reasoning-cycles', str(MAX_REASONING_CYCLES),
    '--termination-threshold', str(TERMINATION_THRESHOLD),
    '--seed', str(SEED),
    '--strict-suite-exec',
    '--no-reuse-existing-suite-artifacts',
], check=False)
print('freeze rc =', freeze_proc.returncode)


In [ ]:
# 6) Full-run diff against frozen baseline.
shutil.rmtree(REPORT_DIFF_DIR, ignore_errors=True)

diff_proc = run([
    'scripts/phase_d_gate.py',
    '--phase', 'D',
    '--device', DEVICE,
    '--model-run-dir', str(H100_UNINTERRUPTED),
    '--phase-root', str(PHASE_ROOT),
    '--baseline-report-dir', str(BASELINE_DIR),
    '--output-dir', str(REPORT_DIFF_DIR),
    '--conceptarc-corpus', 'tools/ConceptARC/corpus',
    '--rearc-tasks', 'data/arc/re_arc/tasks',
    '--h100-uninterrupted', str(H100_UNINTERRUPTED),
    '--h100-execute', str(H100_EXECUTE),
    '--h100-pre-commit', str(H100_PRE_COMMIT),
    '--h100-post-commit', str(H100_POST_COMMIT),
    '--max-reasoning-cycles', str(MAX_REASONING_CYCLES),
    '--termination-threshold', str(TERMINATION_THRESHOLD),
    '--seed', str(SEED),
    '--strict-suite-exec',
    '--no-reuse-existing-suite-artifacts',
], check=False)
print('diff rc =', diff_proc.returncode)


In [ ]:
# 7) Inspect summary report.
summary_path = REPORT_DIFF_DIR / 'summary_report.json'
if not summary_path.exists():
    summary_path = REPORT_FREEZE_DIR / 'summary_report.json'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('summary path:', summary_path)
print('regression.status =', summary.get('regression.status'))
print('suite_status =', json.dumps(summary.get('suite_status', {}), indent=2, ensure_ascii=False))
print('violations =', json.dumps(summary.get('regression.violations', []), indent=2, ensure_ascii=False))
print('checklist =', json.dumps(summary.get('completion_checklist', {}), indent=2, ensure_ascii=False))


## Expected Outcome

1. H100 GPU is visible to JAX.
2. Pinned-manifest S8 local/H100 paths are rebuilt.
3. `scripts/phase_d_gate.py` full-run freeze and diff both complete.
4. `summary_report.json` includes Phase D checklist fields and suite statuses.


In [ ]:
zip_base = Path('/kaggle/working/iris_phase_d_h100_fullrun_outputs')
zip_file = zip_base.with_suffix('.zip')
if zip_file.exists():
    zip_file.unlink()

archive_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(ARTIFACT_ROOT))
print('compressed artifact root:', ARTIFACT_ROOT)
print('zip output:', archive_path)
print('size bytes:', Path(archive_path).stat().st_size)
